In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
path_train = "/content/drive/MyDrive/UNSW Datasets/UNSW_NB15_testing-set.csv"
df_train = pd.read_csv(path_train)

path_test = "/content/drive/MyDrive/UNSW Datasets/UNSW_NB15_training-set.csv"
df_test = pd.read_csv(path_test)

In [ ]:
print(df_train.shape)
print(df_test.shape)

(175341, 45)
(82332, 45)


In [ ]:
df = pd.concat([df_train, df_test], axis=0)
# df = df_test
df = df.sample(frac=1)

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df['label'].value_counts()

,count
label,
1,164673
0,93000


In [ ]:
df = df.groupby('label')
df = df.sample(n=df.size().min(), replace=True)
df = df.reset_index(drop=True)

In [ ]:
df['label'].value_counts()

,count
label,
0,93000
1,93000


In [ ]:
# list = [i for i in range(11) if i % 2 == 0]
df_columns = df.columns
df_num_columns = df.describe().columns
df_cat_columns = [col for col in df_columns if col not in df_num_columns]

In [ ]:
df_cat_columns

['proto', 'service', 'state', 'attack_cat']

In [ ]:
df.proto.value_counts()

,count
proto,
tcp,98851
udp,61675
unas,8814
arp,3836
ospf,1913
...,...
encap,59
micp,55
igmp,44


In [ ]:
def labelEncoder(dataframe, col_name_list):
    from sklearn import preprocessing
    label_encoder = preprocessing.LabelEncoder()
    encoding_dict = {}

    for col_name in col_name_list:
        dataframe[col_name] = label_encoder.fit_transform(dataframe[col_name])
        # Store the mapping for this column
        encoding_dict[col_name] = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

    return dataframe, encoding_dict

# Apply the function to your datasets
df, encoding_dict_train = labelEncoder(df, df_cat_columns)
df_test, encoding_dict_test = labelEncoder(df_test, df_cat_columns)

In [ ]:
encoding_dict_train

{'proto': {'3pc': np.int64(0),
  'a/n': np.int64(1),
  'aes-sp3-d': np.int64(2),
  'any': np.int64(3),
  'argus': np.int64(4),
  'aris': np.int64(5),
  'arp': np.int64(6),
  'ax.25': np.int64(7),
  'bbn-rcc': np.int64(8),
  'bna': np.int64(9),
  'br-sat-mon': np.int64(10),
  'cbt': np.int64(11),
  'cftp': np.int64(12),
  'chaos': np.int64(13),
  'compaq-peer': np.int64(14),
  'cphb': np.int64(15),
  'cpnx': np.int64(16),
  'crtp': np.int64(17),
  'crudp': np.int64(18),
  'dcn': np.int64(19),
  'ddp': np.int64(20),
  'ddx': np.int64(21),
  'dgp': np.int64(22),
  'egp': np.int64(23),
  'eigrp': np.int64(24),
  'emcon': np.int64(25),
  'encap': np.int64(26),
  'etherip': np.int64(27),
  'fc': np.int64(28),
  'fire': np.int64(29),
  'ggp': np.int64(30),
  'gmtp': np.int64(31),
  'gre': np.int64(32),
  'hmp': np.int64(33),
  'i-nlsp': np.int64(34),
  'iatp': np.int64(35),
  'ib': np.int64(36),
  'icmp': np.int64(37),
  'idpr': np.int64(38),
  'idpr-cmtp': np.int64(39),
  'idrp': np.int64(40

In [ ]:
def labelEncoder(dataframe, col_name_list):
  from sklearn import preprocessing
  label_encoder = preprocessing.LabelEncoder()

  for col_name in col_name_list:
    dataframe[col_name]= label_encoder.fit_transform(dataframe[col_name])

  return dataframe

df = labelEncoder(df, df_cat_columns)
df_test = labelEncoder(df_test, df_cat_columns)

In [ ]:
X = df.drop(['attack_cat', 'label'], axis = 1)
y = df['label']
y_mul = df['attack_cat']

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler = MinMaxScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, shuffle= True, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42, shuffle= True, stratify=y_test)

In [ ]:
X_train.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)

In [ ]:
y_train

,label
0,0
1,0
2,1
3,0
4,0
...,...
148795,0
148796,0
148797,1
148798,1


# 100% Data (Supervised model)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = RandomForestClassifier(random_state=0)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

0.9901075268817204


In [ ]:
# y_pred = model.predict(X_val)
# accuracy = accuracy_score(y_val, y_pred)
# print(accuracy)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score

classification_report = classification_report(y_test, y_pred, digits=4)
confusion_matrix = confusion_matrix(y_test, y_pred)
auc_score = roc_auc_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Classification report:", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

Accuracy: 0.9901075268817204
Classification report:               precision    recall  f1-score   support

           0     0.9881    0.9922    0.9901      9300
           1     0.9921    0.9881    0.9901      9300

    accuracy                         0.9901     18600
   macro avg     0.9901    0.9901    0.9901     18600
weighted avg     0.9901    0.9901    0.9901     18600

Confusion matrix: [[9227   73]
 [ 111 9189]]
AUC score: 0.9901075268817204


In [ ]:
!pip install scikit-activeml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 20.8 MB/s eta 0:00:00


In [ ]:
import numpy as np
import matplotlib as mlp
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.datasets import make_classification

from skactiveml.classifier import SklearnClassifier
from skactiveml.pool import UncertaintySampling
from skactiveml.utils import unlabeled_indices, labeled_indices, MISSING_LABEL
from skactiveml.visualization import plot_decision_boundary, plot_utilities

import warnings
mlp.rcParams["figure.facecolor"] = "white"
warnings.filterwarnings("ignore")

In [ ]:
y_miss = np.full(shape=y_train.shape, fill_value=MISSING_LABEL)
print(y_miss)

[nan nan nan ... nan nan nan]


In [ ]:
clf = SklearnClassifier(RandomForestClassifier(), classes=np.unique(y_train))

In [ ]:
qs = UncertaintySampling(method='entropy', random_state=42)

In [ ]:
n_cycles = 20
y_miss = np.full(shape=y_train.shape, fill_value=MISSING_LABEL)
clf.fit(X_train, y_miss)

SklearnClassifier(classes=array([0, 1]), estimator=RandomForestClassifier())

In [ ]:
int(X_train.shape[0]*.20)

29760

**20%**

In [ ]:
num_labeled = int(X_train.shape[0]*.20)
for c in range(num_labeled):                                                  # Labeled data 1000
    query_idx = qs.query(X=X_train, y=y_miss, clf=clf, batch_size=100)
    # print(query_idx)
    y_miss[query_idx] = y_train[query_idx]
    clf.fit(X_train, y_miss)

In [ ]:
y_pred_ac = clf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

classification_report = classification_report(y_test, y_pred_ac, digits=4)
confusion_matrix = confusion_matrix(y_test, y_pred_ac)
auc_score = roc_auc_score(y_test, y_pred_ac)

print("Classification report:", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

In [ ]:
clf.fit(X_train.to_numpy()[:num_labeled], y_train.to_numpy()[:num_labeled])

In [ ]:
y_pred_1000 = clf.predict(X_test)

import sklearn.metrics
from sklearn.metrics import accuracy_score, classification_report

classification_report = classification_report(y_test, y_pred_1000, digits=4)
confusion_matrix = sklearn.metrics.confusion_matrix(y_test, y_pred_1000)
auc_score = roc_auc_score(y_test, y_pred_1000)

print("Classification report:", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

##Code 50%

In [ ]:
clf = SklearnClassifier(RandomForestClassifier(), classes=np.unique(y_train))
qs = UncertaintySampling(method='entropy', random_state=42)

n_cycles = 20
y_miss = np.full(shape=y_train.shape, fill_value=MISSING_LABEL)
clf.fit(X_train, y_miss)

num_labeled = int(X_train.shape[0]*.50)
for c in range(num_labeled):
    query_idx = qs.query(X=X_train, y=y_miss, clf=clf, batch_size=100)
    # print(query_idx)
    y_miss[query_idx] = y_train[query_idx]
    clf.fit(X_train, y_miss)

y_pred_ac_100 = clf.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

classification_report = classification_report(y_test, y_pred_ac_100, digits=4)
confusion_matrix = confusion_matrix(y_test, y_pred_ac_100)
auc_score = roc_auc_score(y_test, y_pred_ac_100)

print("Classification report:", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

clf.fit(X_train.to_numpy()[:num_labeled], y_train.to_numpy()[:num_labeled])
y_pred_100 = clf.predict(X_test)

import sklearn.metrics
from sklearn.metrics import accuracy_score, classification_report

classification_report = classification_report(y_test, y_pred_100, digits=4)
confusion_matrix = sklearn.metrics.confusion_matrix(y_test, y_pred_100)
auc_score = roc_auc_score(y_test, y_pred_100)

print("Classification report:", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

**5%**

In [ ]:
clf = SklearnClassifier(RandomForestClassifier(), classes=np.unique(y_train))
qs = UncertaintySampling(method='entropy', random_state=42)

n_cycles = 20
y_miss = np.full(shape=y_train.shape, fill_value=MISSING_LABEL)
clf.fit(X_train, y_miss)

num_labeled = int(X_train.shape[0]*.05)
for c in range(num_labeled):
    query_idx = qs.query(X=X_train, y=y_miss, clf=clf, batch_size=100)
    # print(query_idx)
    y_miss[query_idx] = y_train[query_idx]
    clf.fit(X_train, y_miss)

y_pred_ac_10000 = clf.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

classification_report = classification_report(y_test, y_pred_ac_10000, digits=4)
confusion_matrix = confusion_matrix(y_test, y_pred_ac_10000)
auc_score = roc_auc_score(y_test, y_pred_ac_10000)

print("Classification report:", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

clf.fit(X_train.to_numpy()[:num_labeled], y_train.to_numpy()[:num_labeled])
y_pred_10000 = clf.predict(X_test)

import sklearn.metrics
from sklearn.metrics import accuracy_score, classification_report

classification_report = classification_report(y_test, y_pred_10000, digits=4)
confusion_matrix = sklearn.metrics.confusion_matrix(y_test, y_pred_10000)
auc_score = roc_auc_score(y_test, y_pred_10000)

print("Classification report:", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

**1%**

In [ ]:
clf = SklearnClassifier(RandomForestClassifier(), classes=np.unique(y_train))
qs = UncertaintySampling(method='entropy', random_state=42)

n_cycles = 20
y_miss = np.full(shape=y_train.shape, fill_value=MISSING_LABEL)
clf.fit(X_train, y_miss)

num_labeled = int(X_train.shape[0]*.01)
for c in range(num_labeled):
    query_idx = qs.query(X=X_train, y=y_miss, clf=clf, batch_size=100)
    # print(query_idx)
    y_miss[query_idx] = y_train[query_idx]
    clf.fit(X_train, y_miss)

y_pred_ac_10000 = clf.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

classification_report = classification_report(y_test, y_pred_ac_10000, digits=4)
confusion_matrix = confusion_matrix(y_test, y_pred_ac_10000)
auc_score = roc_auc_score(y_test, y_pred_ac_10000)

print("Classification report:", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

clf.fit(X_train.to_numpy()[:num_labeled], y_train.to_numpy()[:num_labeled])
y_pred_10000 = clf.predict(X_test)

import sklearn.metrics
from sklearn.metrics import accuracy_score, classification_report

classification_report = classification_report(y_test, y_pred_10000, digits=4)
confusion_matrix = sklearn.metrics.confusion_matrix(y_test, y_pred_10000)
auc_score = roc_auc_score(y_test, y_pred_10000)

print("Classification report:", classification_report)
print("Confusion matrix:", confusion_matrix)
print("AUC score:", auc_score)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
# Create a figure and axes object.
fig, ax = plt.subplots()

fpr, tpr, thresholds = roc_curve(y_test, y_pred)
fpr_ac, tpr_ac, thresholds = roc_curve(y_test, y_pred_ac)
fpr_1000, tpr_1000, thresholds = roc_curve(y_test, y_pred_1000)
# Plot the ROC curves.
ax.plot(fpr, tpr, label='Model 1')
ax.plot(fpr_ac, tpr_ac, label='Model 2')
ax.plot(fpr_1000, tpr_1000, label='Model 3')

# Set the title and axes labels.
ax.set_title('ROC-AUC Curve')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')

# Add a legend.
ax.legend()

# Show the plot.
plt.show()